In [1]:
!pip install -q langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.2 MB/s eta 0:00:00


In [2]:
!pip install -q langchain langchain-groq

In [3]:
import os
import pandas as pd

from langchain_core.tools import tool
from langchain_groq import ChatGroq

In [6]:
import pandas as pd

df = pd.read_csv("students.csv")

df

,student_id,name,department,python,database,ai,web
0,22CS045,Dhanushya,Computer Science,85,72,90,78
1,22CS046,Rahul,Computer Science,65,70,68,72
2,22CS047,Priya,Information Technology,92,88,95,90
3,22CS048,Arun,Information Technology,55,60,58,62
4,22CS049,Meena,Computer Science,78,85,80,88


In [7]:
import sqlite3

conn = sqlite3.connect("students.db")

df.to_sql(
    "students",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

In [8]:
conn = sqlite3.connect("students.db")

result = pd.read_sql_query(
    "SELECT * FROM students",
    conn
)

conn.close()

result

,student_id,name,department,python,database,ai,web
0,22CS045,Dhanushya,Computer Science,85,72,90,78
1,22CS046,Rahul,Computer Science,65,70,68,72
2,22CS047,Priya,Information Technology,92,88,95,90
3,22CS048,Arun,Information Technology,55,60,58,62
4,22CS049,Meena,Computer Science,78,85,80,88


In [10]:
@tool
def get_student_info(student_id: str):
    """
    Get the name and department of a student using the student ID.
    """
    conn = sqlite3.connect("students.db")

    query = """
    SELECT name, department
    FROM students
    WHERE student_id = ?
    """

    result = pd.read_sql_query(
        query,
        conn,
        params=(student_id,)
    )

    conn.close()

    if result.empty:
        return f"No student found with ID {student_id}"

    return result.iloc[0].to_dict()

In [11]:
@tool
def get_student_marks(student_id: str):
    """
    Get the Python, Database, AI, and Web marks of a student using the student ID.
    """
    conn = sqlite3.connect("students.db")

    query = """
    SELECT python, database, ai, web
    FROM students
    WHERE student_id = ?
    """

    result = pd.read_sql_query(
        query,
        conn,
        params=(student_id,)
    )

    conn.close()

    if result.empty:
        return f"No student found with ID {student_id}"

    return result.iloc[0].to_dict()

In [12]:
@tool
def calculator(expression: str):
    """
    Calculate a mathematical expression such as total marks or average marks.
    """
    try:
        return eval(expression)
    except Exception as e:
        return f"Could not calculate the expression: {e}"

In [13]:
@tool
def get_passing_rules():
    """
    Get the university passing rules.
    """
    return {
        "minimum_overall_average": 40,
        "minimum_mark_each_subject": 35
    }

In [14]:
tools = [
    get_student_info,
    get_student_marks,
    calculator,
    get_passing_rules
]

In [16]:
os.environ["GROQ_API_KEY"] = ""

In [17]:
llm = ChatGroq(
    temperature=0,
    max_tokens=1000,
    timeout=10,
    model_name="openai/gpt-oss-20b"
)

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools
)

In [20]:
question = input("Ask something: ")

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ]
})

print(response["messages"][-1].content)

Ask something: What is the name and department of student 22CS045?
**Student 22CS045**  
- **Name:** Dhanushya  
- **Department:** Computer Science


In [21]:
question = input("Ask something: ")

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ]
})

print(response["messages"][-1].content)

Ask something: What are the marks of 22CS047?
Here are the marks for student **22CS047**:

| Subject | Marks |
|---------|-------|
| Python  | 92 |
| Database | 88 |
| AI | 95 |
| Web | 90 |

Let me know if you’d like any additional calculations (e.g., total or average) or if you need more details!


In [22]:
question = input("Ask something: ")

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ]
})

print(response["messages"][-1].content)

Ask something: What is the total and average mark of 22CS045?
**Student ID:** 22CS045  
**Marks:**  
- Python: 85  
- Database: 72  
- AI: 90  
- Web: 78  

**Total Marks:** 325  
**Average Marks:** 81.25  

Let me know if you need any further calculations or details!
